# Export one raster directly as strict native-pixel HTML

Place the GeoTIFF in the notebook's working folder, or paste its full path when prompted. The notebook reads the original raster block by block and exports every valid flooded source cell as an exact georeferenced polygon. It does not create or display a preview, downsample the data, interpolate an image, use an image overlay, import project `src` code, or fall back to another rendering mode.

The resulting HTML can be large and slow to open when the raster contains many flooded pixels, because every visible polygon is an original TIFF cell.

In [ ]:
from pathlib import Path

import folium
import matplotlib as mpl
import numpy as np
import rasterio
from IPython.display import FileLink, display
from pyproj import Transformer

raster_path_text = input('Raster filename or full path: ').strip().strip(chr(34))
RASTER_PATH = Path(raster_path_text).expanduser().resolve()
THRESHOLD_CM = 0.0
MASK_VALUES = (9999,)

if not RASTER_PATH.is_file():
    raise FileNotFoundError(f'Raster not found: {RASTER_PATH}')

HTML_PATH = RASTER_PATH.with_name(f'{RASTER_PATH.stem}_strict_native_pixels.html')
print(f'Reading original native pixels from: {RASTER_PATH}')

In [ ]:
native_pixels = []

with rasterio.open(RASTER_PATH) as src:
    if src.crs is None:
        raise ValueError('The raster has no CRS.')

    source_transform = src.transform
    source_crs = src.crs

    for _, window in src.block_windows(1):
        block = src.read(1, window=window, masked=True)
        values = np.asarray(block.data)
        invalid = np.ma.getmaskarray(block).copy()
        invalid |= ~np.isfinite(values)
        invalid |= values <= THRESHOLD_CM
        for mask_value in MASK_VALUES:
            invalid |= values == mask_value

        rows, cols = np.where(~invalid)
        native_pixels.extend(
            (int(window.row_off + row), int(window.col_off + col), float(values[row, col]))
            for row, col in zip(rows, cols)
        )

if not native_pixels:
    raise ValueError('No valid flooded native pixels were found.')

depths = np.fromiter((pixel[2] for pixel in native_pixels), dtype=np.float64)
vmin = float(depths.min())
vmax = float(np.quantile(depths, 0.995))
if vmax <= vmin:
    vmax = float(depths.max())
if vmax <= vmin:
    vmax = vmin + 1.0

row_values = [pixel[0] for pixel in native_pixels]
col_values = [pixel[1] for pixel in native_pixels]
row_min, row_max = min(row_values), max(row_values) + 1
col_min, col_max = min(col_values), max(col_values) + 1

transformer = Transformer.from_crs(source_crs, 'EPSG:4326', always_xy=True)
extent_x = []
extent_y = []
for col, row in ((col_min, row_min), (col_max, row_min), (col_max, row_max), (col_min, row_max)):
    x, y = source_transform * (col, row)
    extent_x.append(x)
    extent_y.append(y)
extent_lon, extent_lat = transformer.transform(extent_x, extent_y)
map_bounds = [[float(min(extent_lat)), float(min(extent_lon))], [float(max(extent_lat)), float(max(extent_lon))]]
map_center = [(map_bounds[0][0] + map_bounds[1][0]) / 2, (map_bounds[0][1] + map_bounds[1][1]) / 2]

strict_map = folium.Map(location=map_center, zoom_start=9, tiles='CartoDB positron', prefer_canvas=True)
cmap = mpl.colormaps['turbo']
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax, clip=True)

for row, col, value in native_pixels:
    corners = (
        source_transform * (col, row),
        source_transform * (col + 1, row),
        source_transform * (col + 1, row + 1),
        source_transform * (col, row + 1),
    )
    xs, ys = zip(*corners)
    lons, lats = transformer.transform(xs, ys)
    folium.Polygon(
        locations=list(zip(lats, lons)),
        stroke=False,
        fill=True,
        fill_color=mpl.colors.to_hex(cmap(norm(value))),
        fill_opacity=0.9,
        tooltip=f'Depth: {value:.1f} cm',
    ).add_to(strict_map)

strict_map.fit_bounds(map_bounds)
strict_map.save(str(HTML_PATH))

print(f'Exported {len(native_pixels):,} original native pixels.')
print(f'HTML saved to: {HTML_PATH}')
display(FileLink(str(HTML_PATH), result_html_prefix='Download HTML map: '))